# Lab 4: Recurrent neural networks (RNNs)

The purpose of this notebook is to learn how to write a RNN from scratch. You will also implement an RNN using the PyTorch library.

Work through the cells below, running each cell in turn. In various places you will see the words "TODO". Follow the instructions at these places and write code to complete the functions.

This blog provides a nice walkthrough of the maths behind backpropagation through time: https://d2l.ai/chapter_recurrent-neural-networks/bptt.html

In [ ]:
# download the "TinyShakespeare" dataset
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

In [ ]:
import numpy as np

# data I/O
data = open('input.txt', 'r').read() # should be simple plain text file
chars = list(set(data))
data_size, vocab_size = len(data), len(chars)
print ('data has %d characters, %d unique.' % (data_size, vocab_size))
char_to_ix = { ch:i for i,ch in enumerate(chars) }
ix_to_char = { i:ch for i,ch in enumerate(chars) }

# hyperparameters
hidden_size = 100 # size of hidden layer of neurons
seq_length = 25 # number of steps to unroll the RNN for
learning_rate = 1e-1

In [ ]:
# model parameters
Wxh = np.random.randn(hidden_size, vocab_size)*0.01 # input to hidden
Whh = np.random.randn(hidden_size, hidden_size)*0.01 # hidden to hidden
Why = np.random.randn(vocab_size, hidden_size)*0.01 # hidden to output
bh = np.zeros((hidden_size, 1)) # hidden bias
by = np.zeros((vocab_size, 1)) # output bias

def lossFun(inputs, targets, hprev):
  """
  inputs,targets are both list of integers.
  hprev is Hx1 array of initial hidden state
  returns the loss, gradients on model parameters, and last hidden state
  """
  xs, hs, ys, ps = {}, {}, {}, {}
  hs[-1] = np.copy(hprev)
  loss = 0
  # forward pass
  for t in range(len(inputs)):
    xs[t] = np.zeros((vocab_size,1)) # encode in 1-of-k representation
    xs[t][inputs[t]] = 1
    hs[t] = #TODO # hidden state
    ys[t] = np.dot(Why, hs[t]) + by # unnormalized log probabilities for next chars
    ps[t] = np.exp(ys[t]) / np.sum(np.exp(ys[t])) # probabilities for next chars
    loss += #TODO # softmax (cross-entropy loss)
  # backward pass: compute gradients going backwards
  dWxh, dWhh, dWhy = np.zeros_like(Wxh), np.zeros_like(Whh), np.zeros_like(Why)
  dbh, dby = np.zeros_like(bh), np.zeros_like(by)
  dhnext = np.zeros_like(hs[0])

  for t in reversed(range(len(inputs))):
    dy = np.copy(ps[t])
    dy[targets[t]] -= 1 # backprop into y: dL_{t}/dy_{t} (if confused here see https://cs231n.github.io/neural-networks-case-study/#grad)
    dWhy += #TODO # dL_{t}/dWhy = dL_{t}/dy_{t} * dy_{t}/dWhy
    dby += dy
    dh = np.dot(Why.T, dy) + dhnext # backprop into h: dL_{t}/dh_{t} = dL_{t}/dy_{t} * dy_{t}/dh_{t} + dL_{t}/dh_{t+1} * dh_{t+1}/dh_{t}
    dhraw = #TODO # backprop through tanh nonlinearity: dL_{t}/du_{t} = dh_{t}/du_{t} * dL_{t}/dh_{t}; h_{t} = tanh(u), u = Whh*h_{t+1} + Wxh*x_{t}
    dbh += dhraw
    dWxh += #TODO # dL_{t}/dWxh = dL_{t}/du_{t} * du_{t}/dWxh
    dWhh += #TODO # dL_{t}/dWhh = dL_{t}/du_{t} * du_{t}/dWhh
    dhnext = np.dot(Whh.T, dhraw) # dL_{t}/dh{t+1} = du_{t}/dh_{t+1} * dL_{t}/du_{t}

  for dparam in [dWxh, dWhh, dWhy, dbh, dby]:
    np.clip(dparam, -5, 5, out=dparam) # clip to mitigate exploding gradients
  return loss, dWxh, dWhh, dWhy, dbh, dby, hs[len(inputs)-1]

def sample(h, seed_ix, n):
  """
  sample a sequence of integers from the model
  h is memory state, seed_ix is seed letter for first time step
  """
  x = np.zeros((vocab_size, 1))
  x[seed_ix] = 1
  ixes = []
  for t in range(n):
    h = np.tanh(np.dot(Wxh, x) + np.dot(Whh, h) + bh)
    y = np.dot(Why, h) + by
    p = np.exp(y) / np.sum(np.exp(y))
    ix = np.random.choice(range(vocab_size), p=p.ravel())
    x = np.zeros((vocab_size, 1))
    x[ix] = 1
    ixes.append(ix)
  return ixes

n, p = 0, 0
mWxh, mWhh, mWhy = np.zeros_like(Wxh), np.zeros_like(Whh), np.zeros_like(Why)
mbh, mby = np.zeros_like(bh), np.zeros_like(by) # memory variables for Adagrad
smooth_loss = -np.log(1.0/vocab_size)*seq_length # loss at iteration 0
while True:
  # prepare inputs (we're sweeping from left to right in steps seq_length long)
  if p+seq_length+1 >= len(data) or n == 0:
    hprev = np.zeros((hidden_size,1)) # reset RNN memory
    p = 0 # go from start of data
  inputs = [char_to_ix[ch] for ch in data[p:p+seq_length]]
  targets = [char_to_ix[ch] for ch in data[p+1:p+seq_length+1]]

  # sample from the model now and then
  if n % 100 == 0:
    sample_ix = sample(hprev, inputs[0], 200)
    txt = ''.join(ix_to_char[ix] for ix in sample_ix)
    print ('----\n %s \n----' % (txt, ))

  # forward seq_length characters through the net and fetch gradient
  loss, dWxh, dWhh, dWhy, dbh, dby, hprev = lossFun(inputs, targets, hprev)
  smooth_loss = smooth_loss * 0.999 + loss * 0.001
  if n % 100 == 0: print ('iter %d, loss: %f' % (n, smooth_loss)) # print progress

  # perform parameter update with Adagrad
  for param, dparam, mem in zip([Wxh, Whh, Why, bh, by],
                                [dWxh, dWhh, dWhy, dbh, dby],
                                [mWxh, mWhh, mWhy, mbh, mby]):
    mem += dparam * dparam
    param += -learning_rate * dparam / np.sqrt(mem + 1e-8) # adagrad update

  p += seq_length # move data pointer
  n += 1 # iteration counter

**Extension task**

Implement a similar model in PyTorch.

In [ ]:
import torch
import torch.nn as nn

class RNN(nn.Module):
    def __init__(self, vocab_size, hidden_size, num_layers=1):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, vocab_size)
        self.rnn = nn.RNN(vocab_size, hidden_size, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, h=None):
        x = self.embed(x)
        out, h = #TODO
        logits = self.fc(out)
        return logits, h

In [ ]:
def train_model(model, steps=20000, lr=1e-3):
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = #TODO
    #TODO write training loop

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

pytorch_rnn = RNN(vocab_size, hidden_size)
train_model(pytorch_rnn)

In [ ]:
import torch.nn.functional as F
# generate samples from trained model
def sample(model, start_char="T", length=200):
    model.eval()
    idx = char_to_ix[start_char]
    input = torch.tensor([[idx]])
    h = None
    result = [start_char]
    for _ in range(length):
        logits, h = model(input, h)
        probs = F.softmax(logits[:, -1], dim=-1)
        idx = torch.multinomial(probs, num_samples=1).item()
        input = torch.tensor([[idx]])
        result.append(ix_to_char[idx])
    return ''.join(result)

In [ ]:
print("Generated by PyTorch RNN")
print(sample(pytorch_rnn, "T"))